# Кандидатогенерация для категории услуг

По запросу надо выбрать из базы до 50 объявлений. Метрика Recall@50.
Решение считает для каждого объявления оценку из четырёх частей (локация, совпадение
слов, микрокатегория, близость эмбеддингов) и берёт 50 лучших

### Данные

In [1]:
import re, time, gc
from pathlib import Path

import numpy as np
import pandas as pd
import scipy.sparse as sp
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.naive_bayes import ComplementNB

DATA = Path("..")
WORK = DATA / "work"
WORK.mkdir(exist_ok=True)

_WS = re.compile(r"[\s\xa0]+")

def normalize(s):
    if s is None:
        return ""
    return _WS.sub(" ", str(s).lower().replace("ё", "е")).strip()

train = pd.read_parquet(DATA / "train.parquet")
bench_queries = pd.read_parquet(DATA / "benchmark_queries.parquet")
bench_items = pd.read_parquet(DATA / "benchmark_items.parquet")

print("train", train.shape)
print("запросы", bench_queries.shape)
print("объявления", bench_items.shape)

train (497673, 19)
запросы (2452, 6)
объявления (189212, 14)


### Что подсказывают данные

3 проверки

In [2]:
print(f"объявлений из базы есть в train {bench_items.item_id.isin(set(train.item_id)).mean():.1%}")
print(f"текстов запросов есть в train  {bench_queries.search_query.isin(set(train.search_query)).mean():.1%}")

объявлений из базы есть в train 9.6%
текстов запросов есть в train  37.0%


Нужен поиск по тексту, так как почти всех объявлений в обучении нет, так что
запомнить ответы не получится

In [3]:
print(f"локация объявления совпала с локацией поиска {(train.item_location_id == train.search_location_id).mean():.1%}")
print(f"запросов, чьей локации нет в базе           {(~bench_queries.search_location_id.isin(set(bench_items.item_location_id))).mean():.1%}")

локация объявления совпала с локацией поиска 83.1%
запросов, чьей локации нет в базе           17.4%


Важно, что локация совпадает почти всегда, но у части запросов такой локации в базе
нет, поэтому для них прямое сравнение не поможет, поэтому нужна карта переходов

In [4]:
tok = lambda s: set(re.findall(r"\w+", normalize(s)))
sample = train.sample(30_000, random_state=0)
q = sample.search_query.map(tok)
t = sample.item_title_raw.map(tok)
tp = (sample.item_title_raw.fillna("") + " " + sample.item_infm_params_text.fillna("")).map(tok)

print(f"все слова запроса в заголовке            {np.mean([len(a & b) == len(a) for a, b in zip(q, t)]):.1%}")
print(f"все слова в заголовке и параметрах       {np.mean([len(a & b) == len(a) for a, b in zip(q, tp)]):.1%}")
print(f"общих слов нет совсем                    {np.mean([len(a & b) == 0 for a, b in zip(q, tp)]):.1%}")

все слова запроса в заголовке            39.1%
все слова в заголовке и параметрах       50.2%
общих слов нет совсем                    16.1%


Заголовка мало, параметры сильно улучшают. Где общих слов нет вообще, поиск по словам
бесполезный, там нужны эмбеддинги

### Валидация

Проверять идеи отправками нельзя, попыток всего 7. Поэтому своя проверка собирается
из train, там ответы известны.

Объявления берутся из базы бенчмарка, а запросы и ответы к ним из train. Запросы,
попавшие в проверку, удаляются из обучения. Иначе модель их запомнит и цифра
получится обманчивой.

Не все правильные ответы лежат в базе, поэтому здесь выходит выше, чем на платформе.
Сравнивать варианты между собой это не мешает

In [5]:
RANDOM_STATE = 42
N_VAL = 3000
SEEN_SHARE = 0.37

QC = ["search_query", "search_location_id", "search_is_delivery_search",
      "search_infm_params_text", "search_category"]

rng = np.random.default_rng(RANDOM_STATE)
hit = train[train.item_id.isin(set(bench_items.item_id))]

grp = (hit.groupby(QC, dropna=False)["item_id"]
          .apply(lambda s: list(dict.fromkeys(s)))
          .rename("gold").reset_index())

texts = grp.search_query.unique()
rng.shuffle(texts)
unseen = set(texts[: int(len(texts) * (1 - SEEN_SHARE))])
grp["is_seen"] = ~grp.search_query.isin(unseen)

rows = grp.groupby("search_query").search_query.transform("size")
seen_pool = grp[grp.is_seen & (rows > 1)]
unseen_pool = grp[~grp.is_seen]

n_seen = int(N_VAL * SEEN_SHARE)
val = pd.concat([seen_pool.sample(min(n_seen, len(seen_pool)), random_state=RANDOM_STATE),
                 unseen_pool.sample(N_VAL - min(n_seen, len(seen_pool)), random_state=RANDOM_STATE)]
                ).reset_index(drop=True)

vk = set(map(tuple, val[QC].astype(object).values))
tk = list(map(tuple, train[QC].astype(object).values))
drop = np.fromiter((k in vk for k in tk), bool, len(train)) | train.search_query.isin(unseen).values
fit = train[~drop].reset_index(drop=True)

print(f"запросов {len(val):,}, знакомых {int(val.is_seen.sum())}, новых {int((~val.is_seen).sum())}")
print(f"обучение {len(fit):,} строк из {len(train):,}")

запросов 3,000, знакомых 1110, новых 1890
обучение 252,671 строк из 497,673


### Поиск по словам

BM25 считает, насколько текст объявления подходит запросу. Написан вручную, потому что
готовые версии проверяют запросы по очереди, а это слишком медленно для 189 тысяч
объявлений. Своя версия считает всё сразу

In [6]:
def doc_text(df, title_repeat=3, desc_chars=600):
    t = df.item_title_raw.fillna("").map(normalize)
    p = df.item_infm_params_text.fillna("").map(normalize)
    d = df.item_description_raw.fillna("").str.slice(0, desc_chars).map(normalize)
    return ((t + " ") * title_repeat + p + " " + d).values


def query_text(df):
    return (df.search_query.fillna("").map(normalize) + " "
            + df.search_infm_params_text.fillna("").map(normalize)).values


class BM25:
    def __init__(self, k1=1.2, b=0.75):
        self.k1, self.b = k1, b
        self.vec = CountVectorizer(token_pattern=r"\w+", dtype=np.float32)

    def fit(self, docs):
        X = self.vec.fit_transform(docs).tocsr()
        n = X.shape[0]
        dl = np.asarray(X.sum(axis=1)).ravel()
        avgdl = dl.mean()
        df = np.asarray((X > 0).sum(axis=0)).ravel()
        idf = np.log(1.0 + (n - df + 0.5) / (df + 0.5)).astype(np.float32)

        tf = X.data
        rl = np.repeat(dl, np.diff(X.indptr))
        X.data = (tf * (self.k1 + 1.0) / (tf + self.k1 * (1 - self.b + self.b * rl / avgdl))).astype(np.float32)
        self.W = (X @ sp.diags(idf)).tocsr().astype(np.float32)
        return self

    def score_batch(self, queries):
        Q = self.vec.transform(queries)
        Q.data = np.ones_like(Q.data, dtype=np.float32)
        return (Q.tocsr() @ self.W.T).toarray()


def topk(S, k=50):
    k = min(k, S.shape[1])
    idx = np.argpartition(-S, kth=k - 1, axis=1)[:, :k]
    order = np.argsort(-np.take_along_axis(S, idx, axis=1), axis=1)
    return np.take_along_axis(idx, order, axis=1)


def recall(preds, golds, k=50):
    v = [len(set(g) & set(p[:k])) / len(set(g)) for p, g in zip(preds, golds) if len(g)]
    return float(np.mean(v)) if v else 0.0


RESULTS = {}

def report(name, preds, t0=None):
    gold, seen = val.gold.tolist(), val.is_seen.values
    r = recall(preds, gold)
    rs = recall([p for p, m in zip(preds, seen) if m], [g for g, m in zip(gold, seen) if m])
    ru = recall([p for p, m in zip(preds, seen) if not m], [g for g, m in zip(gold, seen) if not m])
    tail = f"  ({time.time() - t0:.0f}s)" if t0 else ""
    print(f"{name:34s} R@50={r:.4f}   знакомые={rs:.4f}  новые={ru:.4f}{tail}")
    RESULTS[name] = r
    return r

### Только слова

In [7]:
ids = bench_items.item_id.values
qt = query_text(val)

bm = BM25().fit(doc_text(bench_items))

def predict_text(batch=250):
    out = []
    for i in range(0, len(qt), batch):
        out.extend(ids[topk(bm.score_batch(qt[i:i + batch]))])
    return out

t = time.time()
report("только слова", predict_text(), t)

только слова                       R@50=0.2668   знакомые=0.2761  новые=0.2613  (5s)


0.266760582010582

Мало. По запросу вроде «маникюр» подходящих объявлений тысячи по всей стране

### Плюс локация

Объявление из своего города получает большой бонус, из карты переходов поменьше,
из соседних мест совсем маленький. Бонусы такие большие нарочно, чтобы свой город
всегда шёл раньше чужого. А между собой объявления сортируются уже по словам.

Чужие города при этом не выбрасываются. Если своих объявлений меньше 50, места в
ответе доберутся соседними, а не останутся пустыми

In [8]:
W_EXACT, W_MAP, W_GEO = 1e6, 5e5, 2e5
RADIUS, MAP_MIN = 50, 0.02

locs = np.sort(bench_items.item_location_id.unique())
loc2i = {l: i for i, l in enumerate(locs)}
doc_loc = bench_items.item_location_id.map(loc2i).values.astype(np.int32)

lat = bench_items.item_latitude.astype(float).values
lon = bench_items.item_longitude.astype(float).values
c = pd.DataFrame({"l": doc_loc, "lat": lat, "lon": lon}).groupby("l")[["lat", "lon"]].median()
clat, clon = c.lat.values.astype(np.float32), c.lon.values.astype(np.float32)

mp = fit.groupby(["search_location_id", "item_location_id"]).size().rename("n").reset_index()
mp["p"] = mp.n / mp.groupby("search_location_id").n.transform("sum")
mp = mp[mp.item_location_id.isin(loc2i)]
mp["j"] = mp.item_location_id.map(loc2i)
locmap = {k: (g.j.values, g.p.values.astype(np.float32))
          for k, g in mp.groupby("search_location_id")[["j", "p"]]}


def geo_bonus(sl):
    b = np.zeros(len(locs), np.float32)
    if sl in locmap:
        j, p = locmap[sl]
        b[j[p >= MAP_MIN]] = W_MAP
    qi = loc2i.get(sl, -1)
    if qi >= 0:
        d = np.hypot((clat - clat[qi]) * 111.0,
                     (clon - clon[qi]) * 111.0 * np.cos(np.radians(clat[qi])))
        b[(d < RADIUS) & (b < W_GEO)] = W_GEO
        b[qi] = W_EXACT
    return b


qlocs = val.search_location_id.values

def predict_geo(batch=200):
    out = []
    for i in range(0, len(qt), batch):
        S = bm.score_batch(qt[i:i + batch])
        for r in range(S.shape[0]):
            S[r] += geo_bonus(qlocs[i + r])[doc_loc]
        out.extend(ids[topk(S)])
    return out

t = time.time()
report("плюс локация", predict_geo(), t)

плюс локация                       R@50=0.8030   знакомые=0.8226  новые=0.7915  (6s)


0.8029734126984128

Главный скачок за всю работу

### Плюс микрокатегория

По тексту запроса можно предсказать, к какому виду услуг он относится, и поднять
объявления оттуда. Берётся не одна категория, а вероятности сразу всех. Тогда модель
сама показывает, насколько она уверена.

В признаках и слова целиком, и куски слов по 3-5 букв. Куски нужны из-за опечаток и
разных форм одного слова

In [9]:
Xtr = query_text(fit)

vw = TfidfVectorizer(token_pattern=r"\w+", ngram_range=(1, 2), min_df=2, sublinear_tf=True)
vc = TfidfVectorizer(analyzer="char_wb", ngram_range=(3, 5), min_df=3,
                     sublinear_tf=True, max_features=400_000)
A = sp.hstack([vw.fit_transform(Xtr), vc.fit_transform(Xtr)]).tocsr()
nb = ComplementNB(alpha=0.3).fit(A, fit.item_microcat_id.values)

mc2col = {m: i for i, m in enumerate(nb.classes_)}
doc_mc = bench_items.item_microcat_id.map(mc2col).fillna(len(nb.classes_)).astype(int).values

B = sp.hstack([vw.transform(qt), vc.transform(qt)]).tocsr()
Pmc = np.hstack([nb.predict_proba(B).astype(np.float32), np.zeros((len(qt), 1), np.float32)])

W_MC = 15.0

def predict_mc(batch=200):
    out = []
    for i in range(0, len(qt), batch):
        S = bm.score_batch(qt[i:i + batch])
        for r in range(S.shape[0]):
            q = i + r
            S[r] += geo_bonus(qlocs[q])[doc_loc] + W_MC * Pmc[q][doc_mc]
        out.extend(ids[topk(S)])
    return out

t = time.time()
report("плюс микрокатегория", predict_mc(), t)

плюс микрокатегория                R@50=0.8299   знакомые=0.8493  новые=0.8186  (7s)


0.8299284391534392

### Плюс эмбеддинги

Эмбеддинг переводит текст в вектор, у близких по смыслу текстов векторы похожи. Это
закрывает случаи, где общих слов нет.

Модель multilingual-e5-small, открытая, работает локально. Для объявления в текст идёт
заголовок дважды, потом параметры и начало описания. Заголовок дважды, потому что он
точнее всего описывает услугу. Фильтры поиска в запрос не добавляются, они размывают
короткий текст.

Готовая модель училась на текстах из интернета и про услуги ничего не знает. Зато в
train лежит полмиллиона пар «что искали и что выбрали», на них модель дообучается под
задачу. Обучение занимает около получаса, потом модель сохраняется в файл и берётся
оттуда

In [10]:
from sentence_transformers import SentenceTransformer

MODEL, SEQ, W_EMB = "intfloat/multilingual-e5-small", 192, 85.0
FT_DIR = WORK / "e5_finetuned"
emb_corpus_path = WORK / "emb_corpus.npy"
emb_val_path = WORK / "emb_val.npy"


def item_text(df):
    return ("passage: " + (df.item_title_raw.fillna("") + ". "
            + df.item_title_raw.fillna("") + ". "
            + df.item_infm_params_text.fillna("").str.slice(0, 300) + ". "
            + df.item_description_raw.fillna("").str.slice(0, 500)).map(normalize)).tolist()


def q_text(df):
    return ("query: " + df.search_query.fillna("").map(normalize)).tolist()


def device():
    import torch
    return "mps" if torch.backends.mps.is_available() else "cpu"


def load_model(path=None):
    m = SentenceTransformer(path or MODEL, device=device())
    m.max_seq_length = SEQ
    return m


if not FT_DIR.exists():
    from datasets import Dataset
    from sentence_transformers import losses, SentenceTransformerTrainer
    from sentence_transformers import SentenceTransformerTrainingArguments

    sub = fit.sample(min(150_000, len(fit)), random_state=0)
    ds = Dataset.from_dict({
        "anchor": ("query: " + sub.search_query.fillna("").map(normalize)).tolist(),
        "positive": item_text(sub),
    })
    base = load_model()
    args = SentenceTransformerTrainingArguments(
        output_dir=str(FT_DIR), num_train_epochs=1, per_device_train_batch_size=64,
        learning_rate=2e-5, warmup_ratio=0.1, logging_steps=200,
        save_strategy="no", report_to=[], dataloader_num_workers=0,
    )
    SentenceTransformerTrainer(model=base, args=args, train_dataset=ds,
                               loss=losses.MultipleNegativesRankingLoss(base)).train()
    base.save(str(FT_DIR))
    del base, ds, sub
    gc.collect()


if emb_corpus_path.exists() and np.load(emb_corpus_path).shape[0] == len(bench_items):
    E = np.load(emb_corpus_path)
else:
    m = load_model(FT_DIR)
    E = m.encode(item_text(bench_items), batch_size=128, normalize_embeddings=True,
                 show_progress_bar=False).astype(np.float32)
    np.save(emb_corpus_path, E)
    del m
    gc.collect()

if emb_val_path.exists() and np.load(emb_val_path).shape[0] == len(val):
    Qv = np.load(emb_val_path)
else:
    m = load_model(FT_DIR)
    Qv = m.encode(q_text(val), batch_size=128, normalize_embeddings=True,
                  show_progress_bar=False).astype(np.float32)
    np.save(emb_val_path, Qv)
    del m
    gc.collect()

print("объявления", E.shape, "запросы", Qv.shape)

/Users/mariasimonova/Downloads/dataset/.venv/lib/python3.9/site-packages/urllib3/__init__.py:35: NotOpenSSLWarning: urllib3 v2 only supports OpenSSL 1.1.1+, currently the 'ssl' module is compiled with 'LibreSSL 2.8.3'. See: https://github.com/urllib3/urllib3/issues/3020
  warnings.warn(


объявления (189212, 384) запросы (3000, 384)


In [11]:
def predict_full(qtexts, qemb, qlocations, probs, batch=200):
    out = []
    for i in range(0, len(qtexts), batch):
        S = bm.score_batch(qtexts[i:i + batch])
        with np.errstate(all="ignore"):
            S += W_EMB * (qemb[i:i + batch] @ E.T)
        for r in range(S.shape[0]):
            q = i + r
            S[r] += geo_bonus(qlocations[q])[doc_loc] + W_MC * probs[q][doc_mc]
        out.extend(ids[topk(S)])
    return out

t = time.time()
report("плюс эмбеддинги", predict_full(qt, Qv, qlocs, Pmc), t)

плюс эмбеддинги                    R@50=0.8875   знакомые=0.9033  новые=0.8783  (8s)


0.8875404761904762

### Разбор ошибок

Где именно теряются объявления, которые не попали в ответ. Два варианта: либо объявление
вообще не рассматривалось, потому что лежит далеко, либо рассматривалось, но оказалось
ниже пятидесятого места

In [12]:
id2row = {v: i for i, v in enumerate(ids)}
hit = out_geo = out_rank = 0
ranks = []

for i in range(0, len(qt), 200):
    S = bm.score_batch(qt[i:i + 200])
    with np.errstate(all="ignore"):
        S += W_EMB * (Qv[i:i + 200] @ E.T)
    for r in range(S.shape[0]):
        q = i + r
        gb = geo_bonus(qlocs[q])
        S[r] += gb[doc_loc] + W_MC * Pmc[q][doc_mc]
        top = set(ids[topk(S[r][None, :])[0]])
        for g in val.gold.iloc[q]:
            if g in top:
                hit += 1
            elif g in id2row:
                row = id2row[g]
                if gb[doc_loc[row]] == 0:
                    out_geo += 1
                else:
                    out_rank += 1
                    ranks.append(int((S[r] > S[r][row]).sum()))

total = hit + out_geo + out_rank
print(f"нашли                          {hit / total:.1%}")
print(f"промах, объявление далеко      {out_geo / total:.1%}")
print(f"промах, нашли но не подняли    {out_rank / total:.1%}")
if ranks:
    a = np.array(ranks)
    print(f"\nу вторых медианное место {np.median(a):.0f}")
    print(f"попали бы в топ 200 {np.mean(a < 200):.1%}, в топ 500 {np.mean(a < 500):.1%}")

нашли                          88.7%
промах, объявление далеко      2.7%
промах, нашли но не подняли    8.6%

у вторых медианное место 128
попали бы в топ 200 66.9%, в топ 500 78.8%


Большая часть потерь это вторая группа. Объявление найдено и лежит в нужном городе, но
стоит слишком низко. Чинится это не расширением поиска, а качеством сравнения текстов,
поэтому основные силы ушли в эмбеддинги и их дообучение, а не в границы по локации.

На первую группу пробовала отдавать часть из 50 мест объявлениям без учёта локации.
Стало хуже, поэтому в решении этого нет

### Итог замеров

In [13]:
pd.DataFrame({"Recall@50": RESULTS}).round(4)

,Recall@50
только слова,0.2668
плюс локация,0.8030
плюс микрокатегория,0.8299
плюс эмбеддинги,0.8875


### Ответ

То же самое, только карта локаций и классификатор теперь обучены на всём train.
Прятать от них ничего не надо, это уже не проверка, а итоговый расчёт

In [14]:
tr_full = train

mp = tr_full.groupby(["search_location_id", "item_location_id"]).size().rename("n").reset_index()
mp["p"] = mp.n / mp.groupby("search_location_id").n.transform("sum")
mp = mp[mp.item_location_id.isin(loc2i)]
mp["j"] = mp.item_location_id.map(loc2i)
locmap = {k: (g.j.values, g.p.values.astype(np.float32))
          for k, g in mp.groupby("search_location_id")[["j", "p"]]}

Xall = query_text(tr_full)
vw = TfidfVectorizer(token_pattern=r"\w+", ngram_range=(1, 2), min_df=2, sublinear_tf=True)
vc = TfidfVectorizer(analyzer="char_wb", ngram_range=(3, 5), min_df=3,
                     sublinear_tf=True, max_features=400_000)
A = sp.hstack([vw.fit_transform(Xall), vc.fit_transform(Xall)]).tocsr()
nb = ComplementNB(alpha=0.3).fit(A, tr_full.item_microcat_id.values)

mc2col = {m: i for i, m in enumerate(nb.classes_)}
doc_mc = bench_items.item_microcat_id.map(mc2col).fillna(len(nb.classes_)).astype(int).values

qt_bench = query_text(bench_queries)
Bb = sp.hstack([vw.transform(qt_bench), vc.transform(qt_bench)]).tocsr()
Pb = np.hstack([nb.predict_proba(Bb).astype(np.float32), np.zeros((len(qt_bench), 1), np.float32)])

emb_q_path = WORK / "emb_bench_q.npy"
if emb_q_path.exists() and np.load(emb_q_path).shape[0] == len(bench_queries):
    Qb = np.load(emb_q_path)
else:
    m = load_model(FT_DIR)
    Qb = m.encode(q_text(bench_queries), batch_size=128, normalize_embeddings=True,
                  show_progress_bar=False).astype(np.float32)
    np.save(emb_q_path, Qb)
    del m
    gc.collect()

t = time.time()
preds = predict_full(qt_bench, Qb, bench_queries.search_location_id.values, Pb)
print(f"готово за {time.time() - t:.0f}s")

готово за 6s


In [15]:
rows = []
for top in preds:
    seen, uniq = set(), []
    for it in top:
        if it not in seen:
            seen.add(it)
            uniq.append(it)
    rows.append(" ".join(uniq[:50]))

answer = pd.DataFrame({"query_id": bench_queries.query_id.astype(str), "answer": rows})
answer.to_csv(DATA / "answer.csv", index=False)

valid = set(bench_items.item_id.astype(str))
lens = answer.answer.str.split(" ").map(len)
allit = [i for r in answer.answer for i in r.split(" ")]

print(f"строк {len(answer):,}, нужно {len(bench_queries):,}")
print(f"колонки {list(answer.columns)}")
print(f"query_id совпадают {set(answer.query_id) == set(bench_queries.query_id.astype(str))}")
print(f"в строке от {lens.min()} до {lens.max()}")
print(f"повторов {sum(len(r.split(' ')) != len(set(r.split(' '))) for r in answer.answer)}")
print(f"нет в базе {sum(1 for i in allit if i not in valid)}")
print(f"длина 16 и строчные {all(len(i) == 16 and i == i.lower() for i in allit)}")
answer.head()

строк 2,452, нужно 2,452
колонки ['query_id', 'answer']
query_id совпадают True
в строке от 50 до 50
повторов 0
нет в базе 0
длина 16 и строчные True


,query_id,answer
0,70DfDUpwjxB4lzFd,355392014208b7bf 9515d1e1ecdac72c 2d73dacac243...
1,JTrdTaZJvSiLPkXj,367af128a9ea2a48 cbeccbecb1fb8d86 422d3ffdd5bb...
2,LZCZNoVG4AFUkVRJ,168a9207e80b0be4 dab52187b4500d9b 3f89b8062dc8...
3,660ac9QVtXkRxZC3,af91ec4a29b66636 bd2cb4500fad728e 1e5924dc546b...
4,YgHcM9MVbxKnxD1e,13f58fe9542bdb1a ba78ad593ec3412d 615c73ea4c3b...


### Что использовано

pandas, numpy, scipy, scikit-learn (CountVectorizer, TfidfVectorizer, ComplementNB),
sentence-transformers, datasets, pyarrow. Всё открытое.

Модель эмбеддингов intfloat/multilingual-e5-small, лицензия MIT, работает локально,
дообучена на train. BM25 написан вручную